**1.IMPORTS**

Here the main thing is to include all the libraries we are going to use in this course of the project 

**-numpy :** to handle the math operation on arrays 

**-pandas :** to handle our csv data set on phising urls 

**-matplotlib :** for plotting our graphs and better visualizations 

**-sklearn :** we use 4 functions from this library:

        -model selection : to split the data into the train and test automatically 

        -linear_model : contains the sigmoid rather than building it from scratch for our logistic regression classifier 1

        -ensemble : to get a decision tree and use in our 2nd classifier 

        -metrics : we get subfunctions for accuracy score  and confusion matrix to proper understand how well our model is doing 

**-xgboost :** same like decision tree but smarter rather than all voting and taking the majority, each tree learns from the mistakes of the previous  

**-warning :** tells python to ignore a lot of warnings in general 


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


Now we access the csv of data and we read into  a table using the data frame. We print out the first 5 rows 

In [2]:
# This makes us load the dataset using the panda library.
df = pd.read_csv('phishing_site_urls.csv')

# See the first 5 rows of the dataset
df.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


We have to get a better understanding of how the data is distributed before building our model the labels, amounts of data whether the data is skewed to one side of the labels or something. In short get a feel of the data before we proceed 


In [3]:
# How many rows and columns do we have in the dataset?
print("Shape:", df.shape)

# What are the column names?
print("Columns:", df.columns.tolist())

# How many of each label do we have?
print("\nLabel counts:")
print(df['Label'].value_counts())

Shape: (549346, 2)
Columns: ['URL', 'Label']

Label counts:
Label
good    392924
bad     156422
Name: count, dtype: int64


We realize the dat shows we have more good emails than the bad ones. We proceed to change the labels from bad or good to 1 and 0 respectfully. This is the turning point in the whole thing right now. This will serve as our real labels on the data hence forth 


In [4]:
# Convert labels to numbers
# bad = 1 (phishing), good = 0 (legit)
df['Label'] = df['Label'].map({'bad': 1, 'good': 0})

print("Labels after conversion:")
print(df['Label'].value_counts())
print("\nFirst 5 rows:")
df.head()

Labels after conversion:
Label
0    392924
1    156422
Name: count, dtype: int64

First 5 rows:


,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,1
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,1
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,1
3,mail.printakid.com/www.online.americanexpress....,1
4,thewhiskeydregs.com/wp-content/themes/widescre...,1


Now we are done changing our labels to binary and all we can now get started to getting the features that play an important role and how we extract them from the URL. The common features associated with phising emails are the following
1. url_length        → how long is the URL? 
2. num_dots          → how many dots? 
3. has_https         → does it use HTTPS? 
4. has_ip            → does it contain an IP address? 
5. num_slashes       → how many slashes? 
6. has_at            → does it have @ symbol?
7. num_subdomains    → how many subdomains? 
8. has_suspicious    → does it contain words like login/verify/secure?
9. url_entropy       → how random does it look? 
10. num_digits       → how many numbers in the URL?